# 02 — Understand the dataset and screen item coverage
 **Purpose:** check the integrated data and identify items with too few training records.
 **Saved:** `training_item_coverage.csv`, including the exclusion decision used by Notebook 03.
 Other summaries stay in this notebook. Save the notebook after running it to retain tables and plots for your report.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pricecatcher_config as cfg

df = pd.read_parquet(cfg.STAGE1_COMBINED_PARQUET)
df["date"] = pd.to_datetime(df["date"])
print("Rows:", len(df), "| Period:", df["date"].min(), "to", df["date"].max())
print("Unique premises:", df["premise_code"].nunique())


## 1. Check variables, invalid prices and duplicate keys


In [ ]:
overview = pd.DataFrame({"dtype": df.dtypes.astype(str),
                         "null_count": df.isna().sum(), "unique_values": df.nunique()})
invalid = df["price"].isna() | ~np.isfinite(df["price"]) | (df["price"] <= 0)
print("Invalid prices:", invalid.sum())
print("Duplicate keys:", df.duplicated(["date", "premise_code", "item_code"]).sum())
display(overview)


## 2. Describe the dataset


In [ ]:
# Full-period counts describe scope; they do not select modelling settings.
display(df.groupby("project_group").size().rename("records").to_frame())
print("States:", df["state"].nunique(), "| Districts:", df["district"].nunique(),
      "| Premise types:", df["premise_type"].nunique(), "| Item codes:", df["item_code"].nunique())
monthly = df.groupby(df["date"].dt.to_period("M")).size()
monthly.plot.bar(figsize=(10, 3), title="Selected food records per month")
plt.ylabel("Records")
plt.tight_layout()
plt.show()


## 3. Screen items using training retail data only
 Exclude items below 5% of the median non-zero item count, and items with no valid training records. This is an item-level screening rule; Notebook 05 checks local weekly coverage separately. It does not guarantee every remaining local series is reliable.


In [ ]:
training = df.loc[(df["date"] <= pd.Timestamp(cfg.TRAIN_END_DATE)) & ~invalid].copy()
if cfg.REMOVE_BORONG:
    training = training.loc[training["premise_type"] != "Borong"].copy()
if training.empty or not 0 < cfg.LOW_COVERAGE_RATIO < 1:
    raise ValueError("Check training dates and the low-coverage ratio.")
counts = training.groupby("item_code").size().reindex(cfg.SCOPE_ITEM_CODES, fill_value=0)
coverage_cutoff = float(counts.loc[counts > 0].median() * cfg.LOW_COVERAGE_RATIO)
item_coverage = counts.rename("training_records").reset_index()
item_coverage["project_group"] = item_coverage["item_code"].map(cfg.PROJECT_ITEM_MAP)
item_coverage["coverage_cutoff"] = coverage_cutoff
item_coverage["exclude_low_coverage"] = (item_coverage["training_records"] == 0) | (item_coverage["training_records"] < coverage_cutoff)
item_coverage["training_end"] = cfg.TRAIN_END_DATE
item_coverage.to_csv(cfg.STAGE2_ITEM_COVERAGE_CSV, index=False)
print("Training-only item cutoff:", round(coverage_cutoff, 2))
print("Excluded item codes:", item_coverage.loc[item_coverage["exclude_low_coverage"], "item_code"].tolist())
display(item_coverage.loc[item_coverage["exclude_low_coverage"]])
print("Next: Notebook 03.")


## 4. Training coverage by food group
Use training data for preparation decisions. Full-period descriptive counts above do not select settings. The 5% item rule is a project heuristic, not a guarantee of reliable local weeks.

In [ ]:
display(training.groupby("project_group").agg(
    records=("price", "size"), items=("item_code", "nunique"),
    premises=("premise_code", "nunique"), observed_days=("date", "nunique")
))
